# Positive NNLS finite-SOE experiment

This notebook tests a structure-aware finite sum of exponentials (SOE) on the completely monotone power-law kernel.

\[
k(t)=\left(1+\frac{t}{\tau}\right)^{-\alpha},\qquad \alpha>0,\;\tau>0.
\]

Decay rates gamma_j are fixed and positive on a logarithmic dictionary. Only non-negative weights are identified:
\[
\min_{w_j\ge0}\|Aw-k\|_2^2,\qquad A_{ij}=e^{-\gamma_jt_i}.
\]

The experiment records kernel-relative error and memory-operator action error. No joint optimization of decay rates is performed.

In [ ]:
import sys
from pathlib import Path
import time
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import nnls

ROOT = Path.cwd()
if not (ROOT / "min").is_dir() and (ROOT.parent / "min").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from min import MemoryOperator, power_law_kernel

In [ ]:
dt = 0.02
t = np.arange(401) * dt
alpha = 0.7
tau = 0.5
kernel_samples = power_law_kernel(t, alpha=alpha, tau=tau)
x = np.sin(2 * np.pi * 0.4 * t) + 0.35 * np.sin(2 * np.pi * 1.7 * t)
reference_output = MemoryOperator(lambda lag: power_law_kernel(lag, alpha=alpha, tau=tau)).apply(t, x)
    gamma_min, gamma_max = 0.05, 20.0
orders = [2, 4, 8, 16]
print(f"samples={t.size}, dt={dt}, horizon={t[-1]:.2f}, alpha={alpha}, tau={tau}")
print(f"positive gamma range=[{gamma_min}, {gamma_max}]")

In [ ]:
def fit_fixed_rate_nnls(samples, t, gamma_min, gamma_max, L):
    gammas = np.geomspace(gamma_min, gamma_max, L)
    A = np.exp(-np.outer(t, gammas))
    start = time.perf_counter()
    weights, residual_norm = nnls(A, samples)
    return weights, gammas, residual_norm, time.perf_counter() - start

def soe_kernel(t, weights, gammas):
    t = np.asarray(t, dtype=float)
    return np.sum(weights[:, None] * np.exp(-gammas[:, None] * t[None, :]), axis=0)

def operator_from_soe(t, x, weights, gammas):
    return MemoryOperator(lambda lag: soe_kernel(lag, weights, gammas)).apply(t, x)

In [ ]:
rows, fits = [], {}
for L in orders:
    weights, gammas, residual_norm, elapsed = fit_fixed_rate_nnls(kernel_samples, t, gamma_min, gamma_max, L)
    fitted_kernel = soe_kernel(t, weights, gammas)
    fitted_output = operator_from_soe(t, x, weights, gammas)
    kernel_rel = np.linalg.norm(kernel_samples - fitted_kernel) / np.linalg.norm(kernel_samples)
    operator_rel = np.linalg.norm(reference_output - fitted_output) / np.linalg.norm(reference_output)
    active = int(np.count_nonzero(weights > 1e-10 * max(1.0, np.max(weights))))
    rows.append((L, active, kernel_rel, operator_rel, elapsed * 1e3, np.all(weights >= 0), np.all(gammas > 0)))
    fits[L] = (weights, gammas, fitted_kernel, fitted_output)

print("L  active   kernel_rel      operator_rel      NNLS_ms   w>=0  gamma>0")
for L, active, ek, em, ms, w_ok, g_ok in rows:
    print(f"{L:2d} {active:7d}   {ek:12.4e}   {em:14.4e}   {ms:8.3f}   {str(w_ok):5s} {g_ok}")

In [ ]:
for _, _, ek, em, _, w_ok, g_ok in rows:
    assert np.isfinite(ek) and np.isfinite(em) and w_ok and g_ok
print("Admissibility checks passed for every fit.")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(t, kernel_samples, label="power-law reference", linewidth=2)
for L in (4, 16):
    ax.plot(t, fits[L][2], label=f"NNLS SOE, L={L}")
ax.set_xlabel("t"); ax.set_ylabel("k(t)")
ax.set_title("Positive NNLS approximation of long-memory kernel")
ax.legend(); ax.grid(True, alpha=0.25)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(t, reference_output, label="reference MIN output", linewidth=2)
for L in (4, 16):
    ax.plot(t, fits[L][3], label=f"NNLS SOE output, L={L}")
ax.set_xlabel("t"); ax.set_ylabel("(Mx)(t)")
ax.set_title("Operator action: reference vs finite SOE")
ax.legend(); ax.grid(True, alpha=0.25)
plt.show()

## Rate-dictionary sensitivity

The positivity constraint does not remove dependence on the chosen rate dictionary. The following diagnostic keeps L=4 fixed and varies only the positive rate range.

In [ ]:
for lo, hi in [(0.02, 20.0), (0.05, 20.0), (0.10, 50.0)]:
    w, g, _, _ = fit_fixed_rate_nnls(kernel_samples, t, lo, hi, L=4)
    kfit = soe_kernel(t, w, g)
    yfit = operator_from_soe(t, x, w, g)
    ek = np.linalg.norm(kernel_samples - kfit) / np.linalg.norm(kernel_samples)
    em = np.linalg.norm(reference_output - yfit) / np.linalg.norm(reference_output)
    print(f"range=[{lo:0.2f}, {hi:0.1f}]  kernel_rel={ek:.4e}  operator_rel={em:.4e}")

## Interpretation and next experiment

This establishes the structure-aware baseline: positive rate dictionary plus NNLS weights.

It should be compared with the previous classical-Prony stress test at matched nominal mode counts. A better NNLS result does not establish universal superiority; it establishes that enforcing known positive-measure structure materially changes the finite-SOE approximation for this target.

Next: compare Prony, Hankel-SVD / matrix pencil / ESPRIT, vector fitting, and positive NNLS under the same targets, horizons, and error metrics before adaptive rate refinement or hierarchical MIN.